# GAVE2 CMRRWNet V7: Resume Validation And Submission

Use this notebook after the original V7 run failed during validation inference on `g_099`. It reuses the completed Drive checkpoints and path reports. It does **not** profile memory, train models, regenerate OOF predictions, or refit path parameters.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

TEAM_ID = "\u68af\u5ea6\u4e0d\u4e0b\u964d\u961f"
DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v7.zip"
EXPECTED_ARCHIVE_SHA256 = "A5927800103F68AC9D4EDE1D03E5E4F1BAB99696A9093F05DCC72C227B47CBF9"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
V6_RUN_DIR = DRIVE_BASE / "runs/gave2_cmrrwnet_v6_3fold"
RUN_DIR = DRIVE_BASE / "runs/gave2_cmrrwnet_v7_3fold"
FOLD_MANIFEST = V6_RUN_DIR / "fold_manifest.json"

V7_VALIDATION_ROOT = RUN_DIR / "predictions/validation"
V7_PATH_VALIDATION_ROOT = RUN_DIR / "predictions/path_validation"
V7_REPORT_ROOT = RUN_DIR / "path_reports"
V6_REFINED_TEAM_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v6_3fold/refined" / TEAM_ID
TASK12_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v7_3fold/task12"
TASK12_TEAM_ROOT = TASK12_OUTPUT_ROOT / TEAM_ID
FINAL_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v7_3fold/main"
FINAL_TEAM_ROOT = FINAL_OUTPUT_ROOT / TEAM_ID
FINAL_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v7_main.zip"
FINAL_REPORT = RUN_DIR / "main_submission_report.json"
AUTO_DISCONNECT = True

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()

def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *[str(value) for value in arguments]]
    print("RUN:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=WORK_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"{module} failed with return code {return_code}")


## Mount Drive And Extract The Corrected Archive


In [ ]:
from google.colab import drive
from pathlib import PurePosixPath

drive.mount("/content/drive")
assert ARCHIVE_PATH.is_file(), f"Missing corrected archive: {ARCHIVE_PATH}"
actual_sha = sha256_file(ARCHIVE_PATH)
assert actual_sha == EXPECTED_ARCHIVE_SHA256, (
    f"Wrong miccai_v7.zip. Expected {EXPECTED_ARCHIVE_SHA256}, found {actual_sha}. "
    "Replace the Drive archive before continuing."
)

required_members = {
    "experiments/gave2_ensemble/predict_v7.py",
    "experiments/gave2_ensemble/path_v7.py",
    "experiments/gave2_ensemble/submission_v7.py",
    "tests/gave2_ensemble/test_v7_core.py",
    "knowledge_base/sources/github/Peng2004_CMRRWNet/train/models.py",
}
anchor = "experiments/gave2_ensemble/predict_v7.py"
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None, "Archive CRC validation failed"
    entries = archive.infolist()
    names = [entry.filename.replace("\\", "/").lstrip("/") for entry in entries]
    matches = [name for name in names if name.endswith(anchor)]
    assert len(matches) == 1, f"Expected one V7 predictor, found {matches}"
    prefix = matches[0][:-len(anchor)]
    relative_names = {name[len(prefix):] for name in names if name.startswith(prefix)}
    missing = sorted(required_members - relative_names)
    assert not missing, f"Corrected archive is missing: {missing}"
    assert any(name.startswith("GAVE2_preliminary/") for name in relative_names)

    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    allowed_roots = {"GAVE2_preliminary", "experiments", "tests", "knowledge_base"}
    root = WORK_ROOT.resolve()
    for entry, normalized in zip(entries, names):
        if not normalized.startswith(prefix):
            continue
        relative = normalized[len(prefix):].lstrip("/")
        if not relative:
            continue
        parts = PurePosixPath(relative).parts
        if not parts or parts[0] not in allowed_roots or ".." in parts:
            continue
        destination = WORK_ROOT.joinpath(*parts)
        resolved = destination.resolve()
        if root != resolved and root not in resolved.parents:
            raise RuntimeError(f"Unsafe archive member: {entry.filename}")
        if entry.is_dir() or relative.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

predictor_source = (WORK_ROOT / "experiments/gave2_ensemble/predict_v7.py").read_text()
assert "repair_ensemble_av_overlap" in predictor_source, "Archive does not contain the ensemble overlap fix"
assert "conditional_av_ensemble_repair_v2" in predictor_source, "Archive cannot invalidate stale validation probabilities"
assert DATA_ROOT.is_dir()
print({"archive_sha256": actual_sha, "members": len(entries), "ensemble_fix": True})


## Verify CUDA And Existing Drive Artifacts


In [ ]:
requirements = WORK_ROOT / "experiments/gave2_ensemble/requirements-gave2-main.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "pytest"], check=True)

import torch

assert torch.cuda.is_available(), "Validation inference requires a CUDA GPU"
assert torch.cuda.is_bf16_supported(), "The trained V7 inference path requires BF16 support"
assert FOLD_MANIFEST.is_file(), f"Missing fold manifest: {FOLD_MANIFEST}"
for task in ("task2", "task1"):
    for fold in range(3):
        fold_root = RUN_DIR / "cmrrwnet_v7" / task / f"fold_{fold}"
        assert (fold_root / "best.pt").is_file(), f"Missing trained checkpoint: {fold_root / 'best.pt'}"
        assert (fold_root / "best.certified.json").is_file()
    assert (V7_REPORT_ROOT / f"{task}_path_gate.json").is_file(), f"Missing path report for {task}"
assert len(list((V6_REFINED_TEAM_ROOT / "Task3").glob("*.txt"))) == 50

subprocess.run(
    [sys.executable, "-m", "pytest", "tests/gave2_ensemble/test_v7_core.py", "-q"],
    cwd=WORK_ROOT,
    check=True,
)
gpu = torch.cuda.get_device_properties(0)
print({"gpu": gpu.name, "vram_gib": round(gpu.total_memory / 1024**3, 2), "bf16": True, "training": "skipped"})


## Resume Validation Inference


In [ ]:
for task in ("task2", "task1"):
    run_module(
        "experiments.gave2_ensemble.predict_v7",
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--store-root", V7_VALIDATION_ROOT / task,
        "--task", task,
        "--mode", "validation",
    )

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.predict_v6 import FloatProbabilityStore
from experiments.gave2_ensemble.predict_v7 import repair_ensemble_av_overlap

for task in ("task2", "task1"):
    store = FloatProbabilityStore(V7_VALIDATION_ROOT / task, task=task, split="validation")
    case_ids = store.list_complete_cases()
    assert len(case_ids) == 50, (task, len(case_ids), case_ids)
    repaired_cases = 0
    for case_id in case_ids:
        probability = store.read_case(case_id)
        repaired = repair_ensemble_av_overlap(probability)
        if bool(((probability[0] >= 0.5) & (probability[2] >= 0.5)).any()):
            metadata = json.loads(store.case_metadata_path(case_id).read_text())
            store.write_case(case_id, repaired, provenance=metadata.get("provenance", {}))
            repaired_cases += 1
    overlaps = 0
    for case_id in case_ids:
        probability = store.read_case(case_id)
        overlaps += int(((probability[0] >= 0.5) & (probability[2] >= 0.5)).sum())
    assert overlaps == 0, f"{task} still has {overlaps} overlapping A/V pixels"
    print({"task": task, "complete_cases": len(case_ids), "repaired_cases": repaired_cases, "av_overlap_pixels": overlaps})


## Apply Existing Path Gates And Write Task 1/2 PNGs


In [ ]:
PATH_REPORTS = {task: V7_REPORT_ROOT / f"{task}_path_gate.json" for task in ("task2", "task1")}
for task in ("task2", "task1"):
    run_module(
        "experiments.gave2_ensemble.path_v7", "apply",
        "--data-root", DATA_ROOT,
        "--source-store-root", V7_VALIDATION_ROOT / task,
        "--output-store-root", V7_PATH_VALIDATION_ROOT / task,
        "--report", PATH_REPORTS[task],
        "--task", task,
        "--source-split", "validation",
    )
    run_module(
        "experiments.gave2_ensemble.path_v7", "promote",
        "--data-root", DATA_ROOT,
        "--source-store-root", V7_PATH_VALIDATION_ROOT / task,
        "--output-root", TASK12_OUTPUT_ROOT,
        "--team-id", TEAM_ID,
        "--task", task,
    )

for task_name in ("Task1", "Task2"):
    files = sorted((TASK12_TEAM_ROOT / task_name).glob("*.png"))
    assert len(files) == 50, (task_name, len(files))
print({task: json.loads(path.read_text())["gate"] for task, path in PATH_REPORTS.items()})


## Assemble And Certify The Main V7 Submission


In [ ]:
from experiments.gave2_ensemble.submission_v6 import readback_zip
from experiments.gave2_ensemble.submission_v7 import assemble_v7, certify_v7

if FINAL_ZIP.exists() or FINAL_REPORT.exists():
    assert FINAL_ZIP.exists() and FINAL_REPORT.exists(), (
        "Only one certified artifact exists. Rename the partial artifact before retrying."
    )
    final_readback = readback_zip(FINAL_ZIP, TEAM_ID)
else:
    if FINAL_TEAM_ROOT.exists():
        shutil.rmtree(FINAL_TEAM_ROOT)
    assemble_v7(TASK12_TEAM_ROOT, V6_REFINED_TEAM_ROOT, FINAL_TEAM_ROOT)
    certify_v7(FINAL_TEAM_ROOT, DATA_ROOT, FINAL_ZIP, FINAL_REPORT)
    final_readback = readback_zip(FINAL_ZIP, TEAM_ID)

assert final_readback["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({
    "ready_to_submit": str(FINAL_ZIP),
    "sha256": final_readback["sha256"],
    "counts": final_readback["counts"],
})


## Disconnect After Certified Readback


In [ ]:
assert FINAL_ZIP.is_file() and FINAL_REPORT.is_file()
assert readback_zip(FINAL_ZIP, TEAM_ID)["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
if AUTO_DISCONNECT:
    print("Certified V7 submission is on Drive. Disconnecting in 10 seconds.", flush=True)
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_DISCONNECT is disabled.")
